# Lick-aligned PSTH & Raster (left vs right licks)

This notebook aligns ephys spike times to **left** and **right** lick events and generates
**separate PSTH + raster figures** for each side, per unit.

- Spikes come from the ephys NWB `units` table (QC-passed units, via `OpticalTagging`).
- Left/right lick times come from the same ephys NWB `acquisition["left_lick_time"|"right_lick_time"]`,
  so they are already in the same clock as the spike times (no synchronization needed).
- Works for a single session or a list of sessions.

In [ ]:
# =============================================================================
# 1. ENVIRONMENT SETUP
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"✅ Analysis modules loaded from: {MODULE_PATH}")
print("🔄 Auto-reload enabled for interactive development")

In [ ]:
# =============================================================================
# 2. IMPORTS, SESSION LIST & HELPER FUNCTIONS
# =============================================================================
import os
import numpy as np
import matplotlib.pyplot as plt

from optical_tagging import OpticalTagging
from behavior_utils import extract_event_timestamps

# ---- Sessions to analyze (add as many as you like) --------------------------
sessions = [
    {
        'behavior_json_file': '/root/capsule/data/ecephys_863973_2026-08-25_14-18-29/behavior/863973_2026-08-25_14-18-29.json',
        'ephys_nwb_file': '/root/capsule/data/ecephys_863973_2026-08-25_14-18-29_sorted_2026-09-09_23-04-54/nwb/ecephys_863973_2026-08-25_14-18-29_experiment1_recording1.nwb',
    },
    # {
    #     'behavior_json_file': '/root/capsule/data/ecephys_XXXXXX_.../behavior/XXXXXX_....json',
    #     'ephys_nwb_file': '/root/capsule/data/ecephys_XXXXXX_..._sorted_.../nwb/ecephys_XXXXXX_..._experiment1_recording1.nwb',
    # },
]

# Where figures are saved
SAVE_ROOT = '/root/capsule/results/lick_aligned/'


def session_label_from_path(behavior_json_file):
    """e.g. '863973_2026-08-25_14-18-29' from the behavior json filename."""
    return os.path.basename(behavior_json_file).replace('.json', '')


def compute_aligned_raster_psth(spike_times, event_times, time_window, bin_size):
    """Align spikes to event_times; return per-event offsets and PSTH mean/SEM.

    Parameters
    ----------
    spike_times : array-like
        Spike timestamps (s) for one unit.
    event_times : array-like
        Event (lick) timestamps (s) to align to.
    time_window : (float, float)
        Start/end offsets (s) around each event.
    bin_size : float
        PSTH bin width (s).

    Returns
    -------
    per_event : list of np.ndarray
        Spike offsets (s, relative to each event) within `time_window`.
    centers : np.ndarray
        PSTH bin centers (s).
    fr : np.ndarray
        Mean firing rate per bin (Hz).
    sem : np.ndarray
        SEM of firing rate per bin (Hz).
    """
    spike_times = np.asarray(spike_times, dtype=float)
    event_times = np.asarray(event_times, dtype=float)
    event_times = event_times[~np.isnan(event_times)]
    event_times = np.sort(event_times)

    bins = np.arange(time_window[0], time_window[1] + bin_size, bin_size)
    centers = bins[:-1] + bin_size / 2.0

    per_event = []
    counts_mat = np.zeros((len(event_times), len(centers)), dtype=float)
    for i, t0 in enumerate(event_times):
        rel = spike_times[(spike_times >= t0 + time_window[0]) &
                          (spike_times <= t0 + time_window[1])] - t0
        per_event.append(rel)
        counts_mat[i], _ = np.histogram(rel, bins=bins)

    if len(event_times) > 0:
        fr = counts_mat.mean(axis=0) / bin_size
    else:
        fr = np.zeros_like(centers)
    if len(event_times) > 1:
        sem = counts_mat.std(axis=0, ddof=1) / np.sqrt(len(event_times)) / bin_size
    else:
        sem = np.zeros_like(centers)
    return per_event, centers, fr, sem


def get_lick_times(opto_tag, side):
    """Return sorted left/right lick times (s) in the spike-time clock."""
    event_name = 'left_lick' if side.lower() == 'left' else 'right_lick'
    lick_times = np.asarray(
        extract_event_timestamps(opto_tag.nwb_ephys_data, event_name), dtype=float
    )
    lick_times = lick_times[~np.isnan(lick_times)]
    return np.sort(lick_times)


def plot_lick_aligned(opto_tag, unit_index, side, lick_times=None,
                      time_window=(-1.0, 1.0), bin_size=0.02,
                      session_label='', save_path=None, save_formats=('png',),
                      max_raster_events=500):
    """Plot a raster (top) + PSTH (bottom) for one unit aligned to one lick side.

    Call once per side to obtain separate left and right figures.
    """
    side = side.lower()
    if unit_index not in opto_tag.units_passing_qc.index:
        print(f"Unit {unit_index} not in QC-passed units; skipping.")
        return None

    if lick_times is None:
        lick_times = get_lick_times(opto_tag, side)
    n_licks = len(lick_times)

    spikes = opto_tag.units_passing_qc.loc[unit_index]["spike_times"]
    per_event, centers, fr, sem = compute_aligned_raster_psth(
        spikes, lick_times, time_window, bin_size
    )

    # Sub-sample events for the raster display only (PSTH still uses all events).
    display_events = per_event
    if max_raster_events is not None and n_licks > max_raster_events:
        idx = np.linspace(0, n_licks - 1, max_raster_events).astype(int)
        display_events = [per_event[i] for i in idx]

    color = 'tab:blue' if side == 'left' else 'tab:red'
    fig, (raster_ax, psth_ax) = plt.subplots(
        2, 1, figsize=(7, 7), sharex=True,
        gridspec_kw={'height_ratios': [3, 1]}
    )
    fig.suptitle(f"{session_label} | Unit {unit_index} | {side} lick ({n_licks} events)", fontsize=12)

    for row, rel in enumerate(display_events):
        raster_ax.vlines(rel, row + 0.5, row + 1.5, color=color, linewidth=0.5)
    raster_ax.axvline(0, color='k', linestyle='--', linewidth=1)
    raster_ax.set_ylabel(f"{side.capitalize()} lick event")
    raster_ax.set_ylim(0.5, max(len(display_events), 1) + 0.5)
    if max_raster_events is not None and n_licks > max_raster_events:
        raster_ax.set_title(f"(showing {max_raster_events} of {n_licks} events)", fontsize=9)

    psth_ax.plot(centers, fr, color=color, label='Mean FR')
    psth_ax.fill_between(centers, fr - sem, fr + sem, color=color, alpha=0.3)
    psth_ax.axvline(0, color='k', linestyle='--', linewidth=1)
    psth_ax.set_xlabel("Time from lick (s)")
    psth_ax.set_ylabel("FR (Hz)")
    psth_ax.set_xlim(time_window)
    psth_ax.legend(loc='upper right')
    fig.tight_layout()

    if save_path:
        os.makedirs(save_path, exist_ok=True)
        base = os.path.join(save_path, f"{session_label}_unit{unit_index}_{side}_lick")
        for fmt in save_formats:
            fig.savefig(f"{base}.{fmt}", dpi=200, bbox_inches='tight')
        print(f"Saved: {base}.[{', '.join(save_formats)}]")

    plt.show()
    return fig


print("Helper functions ready.")

In [ ]:
# =============================================================================
# 3. LOAD A SINGLE SESSION
# =============================================================================
session_idx = 0  # index into `sessions`
session = sessions[session_idx]
session_label = session_label_from_path(session['behavior_json_file'])

OptoTag = OpticalTagging(
    behavior_json_file=session['behavior_json_file'],
    ephys_nwb_file=session['ephys_nwb_file'],
)

left_licks = get_lick_times(OptoTag, 'left')
right_licks = get_lick_times(OptoTag, 'right')
print(f"Session: {session_label}")
print(f"QC-passed units: {len(OptoTag.units_passing_qc)}")
print(f"Left licks: {len(left_licks)} | Right licks: {len(right_licks)}")

In [ ]:
# =============================================================================
# 4. GENERATE SEPARATE LEFT & RIGHT LICK FIGURES FOR SELECTED UNITS
# =============================================================================
# Choose which units to plot. Default: first 5 QC-passed units.
units_to_plot = list(OptoTag.units_passing_qc.index)[:5]
# units_to_plot = [946, 2587]  # or specify your own list

TIME_WINDOW = (-1.0, 1.0)   # seconds around each lick
BIN_SIZE = 0.02             # seconds

save_dir = os.path.join(SAVE_ROOT, session_label)

for unit_index in units_to_plot:
    # LEFT lick figure
    plot_lick_aligned(
        OptoTag, unit_index, side='left', lick_times=left_licks,
        time_window=TIME_WINDOW, bin_size=BIN_SIZE,
        session_label=session_label, save_path=save_dir, save_formats=('png',),
    )
    # RIGHT lick figure (separate figure)
    plot_lick_aligned(
        OptoTag, unit_index, side='right', lick_times=right_licks,
        time_window=TIME_WINDOW, bin_size=BIN_SIZE,
        session_label=session_label, save_path=save_dir, save_formats=('png',),
    )

In [ ]:
# =============================================================================
# 5. (OPTIONAL) BATCH OVER ALL SESSIONS
# =============================================================================
# Generates separate left & right lick figures for every session in `sessions`.
# Set `units_per_session` to control how many units to plot per session, or
# pass an explicit list of unit ids.
units_per_session = 5

for sess in sessions:
    label = session_label_from_path(sess['behavior_json_file'])
    print(f"\n=== Processing {label} ===")
    opto = OpticalTagging(
        behavior_json_file=sess['behavior_json_file'],
        ephys_nwb_file=sess['ephys_nwb_file'],
    )

    l_licks = get_lick_times(opto, 'left')
    r_licks = get_lick_times(opto, 'right')
    print(f"Left licks: {len(l_licks)} | Right licks: {len(r_licks)}")

    units = list(opto.units_passing_qc.index)[:units_per_session]
    out_dir = os.path.join(SAVE_ROOT, label)

    for u in units:
        plot_lick_aligned(opto, u, side='left', lick_times=l_licks,
                          time_window=TIME_WINDOW, bin_size=BIN_SIZE,
                          session_label=label, save_path=out_dir, save_formats=('png',))
        plot_lick_aligned(opto, u, side='right', lick_times=r_licks,
                          time_window=TIME_WINDOW, bin_size=BIN_SIZE,
                          session_label=label, save_path=out_dir, save_formats=('png',))

print("\nAll sessions complete.")